In [22]:
import torch
import torch.nn as nn
import pandas as pd
from transformers import AutoTokenizer, AutoModel
from underthesea import sent_tokenize
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")
phobert = AutoModel.from_pretrained("vinai/phobert-base").to(DEVICE)
class PhoBERTSUM(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Linear(
            encoder.config.hidden_size, 1
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        scores = self.classifier(cls_embeddings)
        return scores
model = PhoBERTSUM(phobert).to(DEVICE)
model.eval()   # inference mode
def split_sentences(text):
    if not isinstance(text, str):
        return []
    sentences = sent_tokenize(text)
    return [s.strip() for s in sentences if len(s.strip()) > 10]
def get_top_k_by_ratio(num_sentences, ratio=0.6):
    return max(1, int(num_sentences * ratio))
def encode_sentences(sentences, max_len=256):
    encoded = tokenizer(
        sentences,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )
    return encoded["input_ids"].to(DEVICE), encoded["attention_mask"].to(DEVICE)
@torch.no_grad()
def extractive_summary(text, ratio=0.6):
    sentences = split_sentences(text)
    n = len(sentences)

    if n == 0:
        return ""

    # Nếu chỉ có 1 câu → lấy luôn
    if n == 1:
        return sentences[0]

    top_k = get_top_k_by_ratio(n, ratio)

    input_ids, attention_mask = encode_sentences(sentences)
    scores = model(input_ids, attention_mask)

    scores = scores.squeeze().cpu()

    # ĐẢM BẢO top_idx LUÔN là list
    top_idx = torch.topk(
        scores, min(top_k, n)
    ).indices.tolist()

    if not isinstance(top_idx, list):
        top_idx = [top_idx]

    top_idx.sort()

    summary = " ".join(sentences[i] for i in top_idx)
    return summary




c:\Users\minhv\anaconda3\envs\pytorch-env\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
INPUT_XLSX = r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\Data TX.xlsx"
OUTPUT_XLSX = r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\Data_TX_with_summary.xlsx"

df = pd.read_excel(INPUT_XLSX)
summaries = []

for idx, row in df.iterrows():
    if pd.isna(row["summary"]) or row["summary"] == "":
        content = row["content"]
        df.at[idx, "summary"] = extractive_summary(content, ratio=0.6)

df.to_excel(OUTPUT_XLSX, index=False)

print("✅ Đã tạo xong bản tóm tắt trích xuất")


✅ Đã tạo xong bản tóm tắt trích xuất
